In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# Step11H.01
# Reload expression matrix and risk information
# ============================================================

import os
import pandas as pd
import numpy as np

project_dir = _hcc_path('')

step11_dir = os.path.join(
    project_dir,
    "LIHC_RNA_Step11_Immune"
)

input_dir = os.path.join(
    step11_dir,
    "Input"
)

result_dir = os.path.join(
    step11_dir,
    "Results"
)

fig_dir = os.path.join(
    step11_dir,
    "Figures"
)


# Expression matrix: Gene × Sample
rna = pd.read_csv(
    os.path.join(
        input_dir,
        "LIHC_Expression_365_log2CPM.tsv"
    ),
    sep="\t",
    index_col=0
)


# RiskScore + RiskGroup + Survival
risk = pd.read_csv(
    os.path.join(
        input_dir,
        "LIHC_RiskGroup_365.csv"
    ),
    index_col=0
)


rna.index = rna.index.astype(str)
rna.columns = rna.columns.astype(str)

risk.index = risk.index.astype(str)


print("RNA:", rna.shape)
print("Risk:", risk.shape)

print("\nRisk columns:")
print(risk.columns.tolist())

print("\nRisk groups:")
print(risk["RiskGroup"].value_counts())

print("\nCommon samples:")
print(len(rna.columns.intersection(risk.index)))

In [ ]:
# ============================================================
# Step11H.02
# Exploratory immunotherapy-related proxy signatures
# ============================================================

immunotherapy_signatures = {

    # 1. IFN-gamma related response
    "IFNG_response": [
        "IFNG",
        "STAT1",
        "IRF1",
        "CXCL9",
        "CXCL10",
        "IDO1"
    ],

    # 2. T-cell inflamed / immune-active proxy
    "T_cell_inflamed": [
        "CD8A",
        "CD8B",
        "CCL5",
        "CXCL9",
        "CXCL10",
        "IFNG",
        "GZMB",
        "PRF1",
        "STAT1",
        "HLA-DRA"
    ],

    # 3. Cytolytic effector activity
    "Cytolytic_effector": [
        "GZMA",
        "GZMB",
        "PRF1",
        "GNLY",
        "NKG7"
    ],

    # 4. Immune checkpoint / exhaustion-related
    "Checkpoint_exhaustion": [
        "PDCD1",
        "CTLA4",
        "LAG3",
        "TIGIT",
        "HAVCR2"
    ],

    # 5. Antigen presentation / MHC activity
    "Antigen_presentation": [
        "B2M",
        "HLA-A",
        "HLA-B",
        "HLA-C",
        "TAP1",
        "TAP2",
        "HLA-DRA"
    ],

    # 6. Immunosuppressive / exclusion-related proxy
    "Immune_suppressive_stroma": [
        "TGFB1",
        "VEGFA",
        "IL10",
        "LGALS9",
        "CD276",
        "VTCN1",
        "FAP",
        "COL1A1",
        "COL1A2"
    ]
}

In [ ]:
# ============================================================
# Step11H.03
# Signature gene coverage
# ============================================================

coverage_rows = []

for sig, genes in immunotherapy_signatures.items():

    present = [
        g for g in genes
        if g in rna.index
    ]

    missing = [
        g for g in genes
        if g not in rna.index
    ]

    coverage_rows.append({
        "Signature": sig,
        "Total_genes": len(genes),
        "Present_genes": len(present),
        "Missing_genes": len(missing),
        "Coverage": len(present) / len(genes),
        "Missing_gene_names": ";".join(missing)
    })


therapy_coverage = pd.DataFrame(
    coverage_rows
)

display(therapy_coverage)

In [ ]:
therapy_coverage.to_csv(
    os.path.join(
        result_dir,
        "LIHC_Immunotherapy_Signature_Gene_Coverage.csv"
    ),
    index=False
)

In [ ]:
# ============================================================
# Step11H.04
# ssGSEA for immunotherapy-related signatures
# ============================================================

import gseapy as gp


therapy_ssgsea = gp.ssgsea(
    data=rna,
    gene_sets=immunotherapy_signatures,

    sample_norm_method="rank",

    outdir=None,
    no_plot=True,

    min_size=2,
    max_size=500,

    permutation_num=0,

    threads=4,
    seed=123
)


print(
    therapy_ssgsea.res2d.shape
)

display(
    therapy_ssgsea.res2d.head()
)

In [ ]:
# ============================================================
# Step11H.05
# Convert ssGSEA output
# ============================================================

therapy_matrix = (
    therapy_ssgsea.res2d
    .pivot(
        index="Name",
        columns="Term",
        values="NES"
    )
)

therapy_matrix.index.name = "Sample_ID"


print(
    "Therapy signature matrix:",
    therapy_matrix.shape
)

print(
    therapy_matrix.columns.tolist()
)

display(
    therapy_matrix.head()
)

In [ ]:
therapy_matrix.to_csv(
    os.path.join(
        result_dir,
        "LIHC_Immunotherapy_Related_Signature_Scores_365.csv"
    )
)

print("Saved.")

In [ ]:
# ============================================================
# Step11H.06
# Merge signatures with RiskScore
# ============================================================

common_samples = (
    therapy_matrix.index
    .intersection(risk.index)
)

therapy_merge = (
    therapy_matrix
    .loc[common_samples]
    .join(
        risk.loc[
            common_samples,
            ["RiskScore", "RiskGroup", "OS_time", "OS"]
        ]
    )
)


print(
    therapy_merge.shape
)

print(
    therapy_merge["RiskGroup"].value_counts()
)

display(
    therapy_merge.head()
)

In [ ]:
# ============================================================
# Step11H.07
# High vs Low comparison
# ============================================================

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


signature_cols = therapy_matrix.columns.tolist()

therapy_group_results = []


for sig in signature_cols:

    high = therapy_merge.loc[
        therapy_merge["RiskGroup"] == "High",
        sig
    ].dropna()

    low = therapy_merge.loc[
        therapy_merge["RiskGroup"] == "Low",
        sig
    ].dropna()


    U, p = mannwhitneyu(
        high,
        low,
        alternative="two-sided"
    )


    therapy_group_results.append({

        "Signature": sig,

        "High_mean": high.mean(),
        "Low_mean": low.mean(),

        "High_median": high.median(),
        "Low_median": low.median(),

        "Median_difference_High_minus_Low":
            high.median() - low.median(),

        "P_value": p
    })


therapy_group_stats = pd.DataFrame(
    therapy_group_results
)


therapy_group_stats["FDR"] = multipletests(
    therapy_group_stats["P_value"],
    method="fdr_bh"
)[1]


therapy_group_stats = (
    therapy_group_stats
    .sort_values(
        ["FDR", "P_value"]
    )
    .reset_index(drop=True)
)


display(
    therapy_group_stats
)

In [ ]:
therapy_group_stats.to_csv(
    os.path.join(
        result_dir,
        "LIHC_Immunotherapy_Signatures_High_vs_Low.csv"
    ),
    index=False
)

In [ ]:
# ============================================================
# Step11H.08
# Correlation with RiskScore
# ============================================================

from scipy.stats import spearmanr


therapy_corr_results = []


for sig in signature_cols:

    temp = therapy_merge[
        [sig, "RiskScore"]
    ].dropna()


    rho, p = spearmanr(
        temp[sig],
        temp["RiskScore"]
    )


    therapy_corr_results.append({

        "Signature": sig,
        "Spearman_rho": rho,
        "P_value": p
    })


therapy_corr_stats = pd.DataFrame(
    therapy_corr_results
)


therapy_corr_stats["FDR"] = multipletests(
    therapy_corr_stats["P_value"],
    method="fdr_bh"
)[1]


therapy_corr_stats = (
    therapy_corr_stats
    .sort_values(
        ["FDR", "P_value"]
    )
    .reset_index(drop=True)
)


display(
    therapy_corr_stats
)

In [ ]:
therapy_corr_stats.to_csv(
    os.path.join(
        result_dir,
        "LIHC_Immunotherapy_Signatures_RiskScore_Correlation.csv"
    ),
    index=False
)

In [ ]:
# ============================================================
# Step11H.09
# Immunotherapy-related signatures: High vs Low
# ============================================================

import matplotlib.pyplot as plt
import os


plot_order = [
    "T_cell_inflamed",
    "Cytolytic_effector",
    "IFNG_response",
    "Antigen_presentation",
    "Checkpoint_exhaustion",
    "Immune_suppressive_stroma"
]


fig, axes = plt.subplots(
    2,
    3,
    figsize=(14, 8)
)

axes = axes.flatten()


for ax, sig in zip(axes, plot_order):

    low = therapy_merge.loc[
        therapy_merge["RiskGroup"] == "Low",
        sig
    ].dropna()

    high = therapy_merge.loc[
        therapy_merge["RiskGroup"] == "High",
        sig
    ].dropna()


    ax.boxplot(
        [low, high],
        labels=["Low", "High"],
        showfliers=False
    )


    row = therapy_group_stats.loc[
        therapy_group_stats["Signature"] == sig
    ].iloc[0]

    fdr = row["FDR"]


    if fdr < 0.001:
        label = "FDR < 0.001"
    else:
        label = f"FDR = {fdr:.3f}"


    ax.text(
        0.5,
        0.95,
        label,
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=9,
        fontweight="bold" if fdr < 0.05 else "normal"
    )


    ax.set_title(
        sig.replace("_", " "),
        fontsize=11
    )

    ax.set_ylabel(
        "ssGSEA NES"
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


fig.suptitle(
    "Immunotherapy-related signatures by risk group",
    fontsize=15
)

fig.tight_layout(
    rect=[0, 0, 1, 0.96]
)


fig.savefig(
    os.path.join(
        fig_dir,
        "Immunotherapy_Signatures_High_vs_Low.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "Immunotherapy_Signatures_High_vs_Low.pdf"
    ),
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11H.10
# Immunotherapy signatures vs RiskScore
# ============================================================

import numpy as np


plot_df = (
    therapy_corr_stats
    .sort_values("Spearman_rho")
    .reset_index(drop=True)
)

y = np.arange(len(plot_df))


fig, ax = plt.subplots(
    figsize=(8.5, 5.5)
)


ax.hlines(
    y=y,
    xmin=0,
    xmax=plot_df["Spearman_rho"],
    linewidth=1.4
)

ax.scatter(
    plot_df["Spearman_rho"],
    y,
    s=75,
    zorder=3
)

ax.axvline(
    0,
    linestyle="--",
    linewidth=1
)


ax.set_yticks(y)

ax.set_yticklabels(
    plot_df["Signature"]
    .str.replace("_", " ")
)


# 给右侧FDR留空间
ax.set_xlim(
    -0.27,
    0.25
)


# rho数值
for i, row in plot_df.iterrows():

    rho = row["Spearman_rho"]

    ax.text(
        rho - 0.008 if rho < 0 else rho + 0.008,
        i,
        f"{rho:.3f}",
        va="center",
        ha="right" if rho < 0 else "left",
        fontsize=8
    )


# FDR固定放右侧
for i, row in plot_df.iterrows():

    fdr = row["FDR"]

    if fdr < 0.001:
        txt = "FDR < 0.001"
    else:
        txt = f"FDR = {fdr:.3f}"

    ax.text(
        0.135,
        i,
        txt,
        va="center",
        ha="left",
        fontsize=9,
        fontweight="bold" if fdr < 0.05 else "normal"
    )


ax.set_xlabel(
    "Spearman correlation with RiskScore"
)

ax.set_title(
    "Immunotherapy-related signatures vs RiskScore",
    fontsize=14
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "Immunotherapy_Signatures_RiskScore_Correlation.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "Immunotherapy_Signatures_RiskScore_Correlation.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11H.11
# Integrate immunotherapy-related signatures + checkpoints
# Correct checkpoint filename
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


checkpoint_file = os.path.join(
    result_dir,
    "LIHC_ImmuneCheckpoint_Expression_365.csv"
)

checkpoint_expr = pd.read_csv(
    checkpoint_file,
    index_col=0
)

checkpoint_expr.index = checkpoint_expr.index.astype(str)


focus_checkpoints = [
    "TNFRSF4",
    "CD276",
    "VTCN1",
    "CD40LG",
    "BTLA",
    "CD160"
]


print("Checkpoint matrix:", checkpoint_expr.shape)

print("\nAvailable focus genes:")
print(
    [g for g in focus_checkpoints
     if g in checkpoint_expr.columns]
)

print("\nMissing focus genes:")
print(
    [g for g in focus_checkpoints
     if g not in checkpoint_expr.columns]
)

display(
    checkpoint_expr.iloc[:5, :8]
)

In [ ]:
# ============================================================
# Step11H.12
# Build integrated immunotherapy-related matrix
# ============================================================

therapy_focus = [
    "T_cell_inflamed",
    "Cytolytic_effector",
    "IFNG_response",
    "Antigen_presentation",
    "Checkpoint_exhaustion",
    "Immune_suppressive_stroma"
]


common_samples = (
    therapy_matrix.index
    .intersection(checkpoint_expr.index)
    .intersection(risk.index)
)


print("Common samples:", len(common_samples))


therapy_part = therapy_matrix.loc[
    common_samples,
    therapy_focus
].copy()


checkpoint_part = checkpoint_expr.loc[
    common_samples,
    focus_checkpoints
].copy()


integrated_therapy = pd.concat(
    [
        therapy_part,
        checkpoint_part
    ],
    axis=1
)


print(
    "Integrated matrix:",
    integrated_therapy.shape
)


integrated_z = (
    integrated_therapy
    -
    integrated_therapy.mean(axis=0)
) / integrated_therapy.std(axis=0)


integrated_z["RiskGroup"] = risk.loc[
    common_samples,
    "RiskGroup"
]


print(
    integrated_z["RiskGroup"].value_counts()
)

display(
    integrated_z.head()
)

In [ ]:
# ============================================================
# Step11H.13
# High vs Low integrated heatmap
# ============================================================

feature_cols = (
    therapy_focus
    +
    focus_checkpoints
)


group_mean = (
    integrated_z
    .groupby("RiskGroup")[feature_cols]
    .mean()
    .T
)


group_mean = group_mean[
    ["Low", "High"]
]


fig, ax = plt.subplots(
    figsize=(6.5, 8)
)


im = ax.imshow(
    group_mean.values,
    aspect="auto"
)


ax.set_xticks(
    np.arange(group_mean.shape[1])
)

ax.set_xticklabels(
    group_mean.columns,
    fontsize=11
)


ax.set_yticks(
    np.arange(group_mean.shape[0])
)

ax.set_yticklabels(
    [
        x.replace("_", " ")
        for x in group_mean.index
    ],
    fontsize=10
)


for i in range(group_mean.shape[0]):

    for j in range(group_mean.shape[1]):

        ax.text(
            j,
            i,
            f"{group_mean.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )


ax.set_title(
    "Integrated immunotherapy-related landscape",
    fontsize=14,
    pad=12
)


fig.colorbar(
    im,
    ax=ax,
    label="Mean Z-score"
)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "Integrated_Immunotherapy_Landscape.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "Integrated_Immunotherapy_Landscape.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11H.14
# Correlation of RiskScore with integrated immunotherapy features
# ============================================================

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import pandas as pd


corr_rows = []


for feature in feature_cols:

    temp = pd.DataFrame({
        "Feature_value": integrated_therapy.loc[
            common_samples,
            feature
        ],

        "RiskScore": risk.loc[
            common_samples,
            "RiskScore"
        ]
    }).dropna()


    rho, p = spearmanr(
        temp["Feature_value"],
        temp["RiskScore"]
    )


    corr_rows.append({
        "Feature": feature,
        "Spearman_rho": rho,
        "P_value": p
    })


therapy_integrated_corr = pd.DataFrame(
    corr_rows
)


therapy_integrated_corr["FDR"] = multipletests(
    therapy_integrated_corr["P_value"],
    method="fdr_bh"
)[1]


therapy_integrated_corr = (
    therapy_integrated_corr
    .sort_values("Spearman_rho")
    .reset_index(drop=True)
)


display(
    therapy_integrated_corr
)

In [ ]:
# ============================================================
# Step11H.15
# Integrated RiskScore correlation plot
# Cleaner version
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import os


plot_df = therapy_integrated_corr.copy()

y = np.arange(len(plot_df))


fig, ax = plt.subplots(
    figsize=(9.5, 7.5)
)


# lollipop lines
ax.hlines(
    y=y,
    xmin=0,
    xmax=plot_df["Spearman_rho"],
    linewidth=1.4
)


# points
ax.scatter(
    plot_df["Spearman_rho"],
    y,
    s=75,
    zorder=3
)


# zero reference
ax.axvline(
    0,
    linestyle="--",
    linewidth=1
)


# y labels
ax.set_yticks(y)

ax.set_yticklabels(
    plot_df["Feature"]
    .str.replace("_", " "),
    fontsize=9
)


# leave room for FDR text
ax.set_xlim(
    -0.40,
    0.45
)


# rho labels beside points
for i, row in plot_df.iterrows():

    rho = row["Spearman_rho"]

    ax.text(
        rho - 0.012 if rho < 0 else rho + 0.012,
        i,
        f"{rho:.3f}",
        va="center",
        ha="right" if rho < 0 else "left",
        fontsize=8
    )


# FDR labels fixed on far right
for i, row in plot_df.iterrows():

    fdr = row["FDR"]

    if fdr < 0.001:
        txt = "FDR < 0.001"
    else:
        txt = f"FDR = {fdr:.3f}"

    ax.text(
        0.27,
        i,
        txt,
        va="center",
        ha="left",
        fontsize=8,
        fontweight="bold" if fdr < 0.05 else "normal"
    )


ax.set_xlabel(
    "Spearman correlation with RiskScore",
    fontsize=11
)

ax.set_title(
    "Integrated immunotherapy-related features vs RiskScore",
    fontsize=14,
    pad=12
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "Integrated_Immunotherapy_RiskScore_Correlation.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "Integrated_Immunotherapy_RiskScore_Correlation.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11H.16
# Save integrated immunotherapy results
# ============================================================

group_mean.to_csv(
    os.path.join(
        result_dir,
        "LIHC_Integrated_Immunotherapy_GroupMean_Zscore.csv"
    )
)

therapy_integrated_corr.to_csv(
    os.path.join(
        result_dir,
        "LIHC_Integrated_Immunotherapy_RiskScore_Correlation.csv"
    ),
    index=False
)

integrated_therapy.to_csv(
    os.path.join(
        result_dir,
        "LIHC_Integrated_Immunotherapy_Features_365.csv"
    )
)

print("Step11H integrated results saved.")